# Plate processing
Run once per plate: set `PLATE` below and run all. Then run `02_compare_plates`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from plate_utils import load_plate, normalize, CONDITION_ORDER
%matplotlib inline

In [ ]:
DATA_DIR = r"C:\Users\sam\Desktop\plate_assay\data"
OUT_DIR = r"C:\Users\sam\Desktop\plate_assay\results"
PLATE = "plate1"   # change to plate2 and run again

In [ ]:
cells, layout = load_plate(DATA_DIR, PLATE)
print(layout["plate"], len(cells), "cells")
cells.head()

## QC
Look at the area histogram. There should be a debris peak at small areas, left of the red line.
**If the debris peak isn't clearly separated from the cells, stop and check with Sam before going on.**

In [ ]:
plt.hist(cells["area"], bins=80)
plt.axvline(35, color="r")
plt.xlabel("area (px)")
plt.show()

In [ ]:
# cut off debris. 35 picked by eye from the histogram (see explore_thresholds.ipynb)
MIN_AREA = 35
cells = cells[cells["area"] >= MIN_AREA]
print(len(cells), "cells after area filter")

In [ ]:
norm = normalize(cells, layout["background"])
norm.to_csv(os.path.join(OUT_DIR, f"normalized_{PLATE}.csv"), index=False)
norm.groupby("condition")["fold_change"].median()

## Bootstrap
Slow (about a minute on my laptop). Don't rerun it just to change the plot.

In [ ]:
# bootstrap 95% CI of the mean fold change per condition
N_BOOT = 200000
rows = []
for cond in CONDITION_ORDER:
    fc = norm.loc[norm["condition"] == cond, "fold_change"].to_numpy()
    means = []
    for i in range(N_BOOT):
        means.append(np.random.choice(fc, size=len(fc), replace=True).mean())
    lo, hi = np.percentile(means, [2.5, 97.5])
    rows.append({"condition": cond, "mean_fc": fc.mean(), "ci_low": lo, "ci_high": hi,
                 "n_cells": len(fc)})
boot = pd.DataFrame(rows)
boot.to_csv(os.path.join(OUT_DIR, f"bootstrap_{PLATE}.csv"), index=False)
boot

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
err = [boot["mean_fc"] - boot["ci_low"], boot["ci_high"] - boot["mean_fc"]]
ax.bar(boot["condition"], boot["mean_fc"], yerr=err, capsize=4, color="#4c72b0")
ax.axhline(1, color="grey", lw=0.8, ls="--")
ax.set_ylabel("fold change vs DMSO")
ax.set_title(layout["plate"])
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"fold_change_{PLATE}.png"), dpi=120)
plt.show()

In [ ]:
print("done", PLATE)